# POSSM Masked Reconstruction Workflow (Stage 1 + Stage 2)

This notebook mirrors the familiar `s5_maskedreconstruction` cadence, but swaps in the POSSM architecture workflow from `possm_ssl`.


In [ ]:
# Mount Drive and resolve cache/output roots.

from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive')
DEFAULT_CACHE_ROOT_SMOOTHED = DRIVE_ROOT / 'utah_ssl' / 'data' / 'cache_v1_smoothed_sigma2p0'
DEFAULT_CACHE_ROOT_RAW = DRIVE_ROOT / 'utah_ssl' / 'data' / 'cache_v1'
OUTPUT_ROOT = DRIVE_ROOT / 'utah_ssl' / 'outputs' / 'ssl_experiments' / 'possm_masked_reconstruction'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('DRIVE_ROOT:', DRIVE_ROOT)
print('DEFAULT_CACHE_ROOT_SMOOTHED exists:', DEFAULT_CACHE_ROOT_SMOOTHED.exists())
print('DEFAULT_CACHE_ROOT_RAW exists:', DEFAULT_CACHE_ROOT_RAW.exists())
print('OUTPUT_ROOT:', OUTPUT_ROOT)


In [ ]:
# Repo import bootstrap (Colab).

import os
import sys
from pathlib import Path

REPO_CANDIDATES = [
    Path('/content/utah-ssl'),
    DRIVE_ROOT / 'utah_ssl' / 'utah-ssl',
]
REPO_ROOT = next((p for p in REPO_CANDIDATES if p.exists()), REPO_CANDIDATES[0])
if not REPO_ROOT.exists():
    raise FileNotFoundError(f'Repo root not found. Checked: {REPO_CANDIDATES}')

EXPERIMENTS_DIR = REPO_ROOT / 'analysis' / 'active' / 'ssl_experiments'
BENCHMARK_DIR = REPO_ROOT / 'analysis' / 'active' / 'transfer_benchmark' / 'ssl_autoresearch'
for path in (REPO_ROOT, EXPERIMENTS_DIR, BENCHMARK_DIR):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

os.chdir(REPO_ROOT)
print('cwd:', Path.cwd())
print('repo root:', REPO_ROOT)


In [ ]:
# Imports.

import json
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from possm_ssl import (
    CacheAccessConfig,
    POSSMFinetuneConfig,
    POSSMTrainingConfig,
    load_precomputed_session_feature_stats_into_cache_context,
    prepare_cache_context,
    recover_possm_run_state_from_checkpoint,
    resolve_possm_checkpoint_path,
    resume_possm_training,
    run_possm_phoneme_finetuning,
    run_possm_training,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)


In [ ]:
# Workflow switches + defaults.

SEED = 7

# Cache selection
USE_SMOOTHED_CACHE = True
ALLOW_RAW_CACHE_FALLBACK = False

# Shared config
FEATURE_MODE = 'tx_sbp'
BOUNDARY_KEY_MODE = 'session'
SEGMENT_BINS = 80
NORMALIZE_IMPL_VERSION = 'session_featurewise_v1'
CACHE_PREPARE_NORMALIZE_IMPL_VERSION = 'segment_prefix_v1'
NORMALIZE_CONTEXT_BINS = min(16, SEGMENT_BINS)
GAUSSIAN_SMOOTHING_SIGMA_BINS = 0.0  # must stay 0.0; smoothing is precomputed in cache
SESSION_STATS_BIN_STRIDE = 2
EXCLUDED_DATASETS = ('brain2text25',)

# Optional precomputed stats (recommended for normalized mode)
SESSION_STATS_VARIANT = 'smoothed'  # one of {'smoothed', 'unsmoothed', 'none'}
SESSION_STATS_PATH_UNSMOOTHED = DRIVE_ROOT / 'utah_ssl' / 'outputs' / 'ssl_experiments' / 'contrastive' / 'precomputed_ssl_session_stats' / 'session_feature_stats_session_featurewise_v1_refds000950_cap126682_tx256_sbp256_stable.pt'
SESSION_STATS_PATH_SMOOTHED = DRIVE_ROOT / 'utah_ssl' / 'outputs' / 'ssl_experiments' / 'contrastive' / 'precomputed_ssl_session_stats' / 'session_feature_stats_session_featurewise_v1_refds000950_cap126682_tx256_sbp256_smooth_sigma2p0_stable.pt'

if SESSION_STATS_VARIANT == 'smoothed':
    STABLE_SESSION_STATS_PATH = SESSION_STATS_PATH_SMOOTHED
elif SESSION_STATS_VARIANT == 'unsmoothed':
    STABLE_SESSION_STATS_PATH = SESSION_STATS_PATH_UNSMOOTHED
elif SESSION_STATS_VARIANT == 'none':
    STABLE_SESSION_STATS_PATH = None
else:
    raise ValueError('SESSION_STATS_VARIANT must be one of {smoothed, unsmoothed, none}')

if STABLE_SESSION_STATS_PATH is not None and not STABLE_SESSION_STATS_PATH.exists():
    print('warning: selected session stats path does not exist; falling back to in-memory stats:', STABLE_SESSION_STATS_PATH)
    STABLE_SESSION_STATS_PATH = None


# Stage-1 temporal backbone
STAGE1_TEMPORAL_BACKBONE_TYPE = 'gru'  # one of {'gru', 'identity'}; keep modular for future S5/Mamba swap
STAGE1_TEMPORAL_GRU_HIDDEN_SIZE = None  # None -> use encoder hidden width
STAGE1_TEMPORAL_GRU_NUM_LAYERS = 1
STAGE1_TEMPORAL_GRU_DROPOUT = 0.0
STAGE1_TEMPORAL_GRU_BIDIRECTIONAL = False
STAGE1_TEMPORAL_BACKBONE_KWARGS = {}  # for future custom registered backbones like S5/Mamba

# Stage-1 objective and masking (default: plain reconstruction)
STAGE1_OBJECTIVE_TYPE = 'plain_mse'  # one of {'plain_mse', 'masked_mse'}
STAGE1_MASKING_TYPE = 'none'  # one of {'none', 'random', 'span', 'channel'}
STAGE1_MASK_PROB = 0.0
STAGE1_MASK_SPAN_BINS = 8
STAGE1_MASK_REPLACE_MODE = 'zero'  # one of {'zero', 'mean', 'gaussian_noise'}

# Stage-1 (reconstruction)
STAGE1_STATE_MODE = 'train'  # one of {'train', 'recover', 'resume'}
STAGE1_DATA_MODE = 'normalized'  # one of {'normalized', 'raw'}
STAGE1_RECOVERY_EXPLICIT_CHECKPOINT_PATH = None
STAGE1_RECOVERY_RUN_DIR = None
STAGE1_RESUME_ADDITIONAL_STEPS = 200

# Stage-2 (phoneme)
STAGE2_STATE_MODE = 'skip'  # one of {'skip', 'probe_frozen', 'finetune_full'}
STAGE2_STAGE1_CHECKPOINT_OVERRIDE = None  # if None, use stage-1 best checkpoint

print({
    'STAGE1_STATE_MODE': STAGE1_STATE_MODE,
    'STAGE1_DATA_MODE': STAGE1_DATA_MODE,
    'STAGE2_STATE_MODE': STAGE2_STATE_MODE,
    'STAGE1_OBJECTIVE_TYPE': STAGE1_OBJECTIVE_TYPE,
    'STAGE1_MASKING_TYPE': STAGE1_MASKING_TYPE,
    'STAGE1_MASK_PROB': STAGE1_MASK_PROB,
    'STAGE1_TEMPORAL_BACKBONE_KWARGS': STAGE1_TEMPORAL_BACKBONE_KWARGS,
    'USE_SMOOTHED_CACHE': USE_SMOOTHED_CACHE,
    'ALLOW_RAW_CACHE_FALLBACK': ALLOW_RAW_CACHE_FALLBACK,
    'FEATURE_MODE': FEATURE_MODE,
    'BOUNDARY_KEY_MODE': BOUNDARY_KEY_MODE,
    'GAUSSIAN_SMOOTHING_SIGMA_BINS': GAUSSIAN_SMOOTHING_SIGMA_BINS,
})


In [ ]:
# Resolve cache root (prefer smoothed by default).

if USE_SMOOTHED_CACHE:
    if DEFAULT_CACHE_ROOT_SMOOTHED.exists():
        CACHE_ROOT = DEFAULT_CACHE_ROOT_SMOOTHED
    elif ALLOW_RAW_CACHE_FALLBACK and DEFAULT_CACHE_ROOT_RAW.exists():
        CACHE_ROOT = DEFAULT_CACHE_ROOT_RAW
        print('warning: smoothed cache missing; falling back to raw cache root')
    else:
        raise FileNotFoundError(
            f'Smoothed cache root not found: {DEFAULT_CACHE_ROOT_SMOOTHED}. '
            'Set ALLOW_RAW_CACHE_FALLBACK=True only if you intentionally want raw cache fallback.'
        )
else:
    CACHE_ROOT = DEFAULT_CACHE_ROOT_RAW

cache_candidates = [CACHE_ROOT]
print('CACHE_ROOT:', CACHE_ROOT)
print('exists:', CACHE_ROOT.exists())
print('datasets:', sorted([p.name for p in CACHE_ROOT.iterdir() if p.is_dir()]) if CACHE_ROOT.exists() else 'missing')


In [ ]:
# Build cache context + optional session stats load.

CACHE_ACCESS_CONFIG = CacheAccessConfig(
    mode='drive_direct',
    local_cache_base='/content/utah_ssl_cache',
    force_recopy_local_cache=False,
    excluded_datasets=EXCLUDED_DATASETS,
    seed=SEED,
    segment_bins=SEGMENT_BINS,
    normalize_context_bins=NORMALIZE_CONTEXT_BINS,
    normalize_impl_version=(
        CACHE_PREPARE_NORMALIZE_IMPL_VERSION
        if (NORMALIZE_IMPL_VERSION == 'session_featurewise_v1' and STABLE_SESSION_STATS_PATH is not None)
        else NORMALIZE_IMPL_VERSION
    ),
    examples_per_shard=8,
    tx_dim=256,
    sbp_dim=256,
    feature_mode=FEATURE_MODE,
    boundary_key_mode=BOUNDARY_KEY_MODE,
    gaussian_smoothing_sigma_bins=GAUSSIAN_SMOOTHING_SIGMA_BINS,
    session_stats_bin_stride=SESSION_STATS_BIN_STRIDE,
)

CACHE_CONTEXT = prepare_cache_context(cache_candidates=cache_candidates, config=CACHE_ACCESS_CONFIG)
print('CACHE_CONTEXT.cache_root:', CACHE_CONTEXT.cache_root)
print('CACHE_CONTEXT.feature_mode:', CACHE_CONTEXT.feature_mode)

SESSION_STATS_STATE = None
if NORMALIZE_IMPL_VERSION == 'session_featurewise_v1' and STABLE_SESSION_STATS_PATH is not None:
    SESSION_STATS_STATE = load_precomputed_session_feature_stats_into_cache_context(
        cache_context=CACHE_CONTEXT,
        stats_path=STABLE_SESSION_STATS_PATH,
        normalize_impl_version=NORMALIZE_IMPL_VERSION,
    )
    print('Loaded precomputed session stats from:', SESSION_STATS_STATE['stats_path'])


In [ ]:
# Preflight smoothing provenance check.

metadata_path = Path(CACHE_CONTEXT.cache_root) / 'brain2text24' / 'metadata.json'
if not metadata_path.exists():
    raise FileNotFoundError(f'Missing metadata for brain2text24: {metadata_path}')

metadata = json.loads(metadata_path.read_text())
smoothing_provenance = metadata.get('smoothing_provenance')
print('smoothing_provenance:', smoothing_provenance)

if USE_SMOOTHED_CACHE:
    if not isinstance(smoothing_provenance, dict):
        raise RuntimeError(
            'USE_SMOOTHED_CACHE=True, but smoothing_provenance missing from dataset metadata. '
            'Point CACHE_ROOT to a smoothed cache or rebuild the smoothed cache root.'
        )
    sigma = smoothing_provenance.get('sigma_bins')
    if sigma is None:
        raise RuntimeError('smoothing_provenance exists but sigma_bins is missing.')

print('Preflight OK.')


In [ ]:
# Stage-1 POSSM config.

STAGE1_CONFIG = POSSMTrainingConfig(
    seed=SEED,
    data_mode=STAGE1_DATA_MODE,
    feature_mode=FEATURE_MODE,
    boundary_key_mode=BOUNDARY_KEY_MODE,
    segment_bins=SEGMENT_BINS,
    model_dim=64,
    latent_count=4,
    value_encoder_type='linear',
    value_mlp_hidden_size=None,
    ffn_hidden_size=256,
    dropout=0.1,
    batch_size=8,
    num_steps=600,
    learning_rate=3e-4,
    weight_decay=1e-2,
    val_every=50,
    val_batches=10,
    checkpoint_every_steps=200,
    dataset_weight_alpha=0.25,
    examples_per_shard=8,
    log_every=10,
    temporal_backbone_type=STAGE1_TEMPORAL_BACKBONE_TYPE,
    temporal_gru_hidden_size=STAGE1_TEMPORAL_GRU_HIDDEN_SIZE,
    temporal_gru_num_layers=STAGE1_TEMPORAL_GRU_NUM_LAYERS,
    temporal_gru_dropout=STAGE1_TEMPORAL_GRU_DROPOUT,
    temporal_gru_bidirectional=STAGE1_TEMPORAL_GRU_BIDIRECTIONAL,
    temporal_backbone_kwargs=STAGE1_TEMPORAL_BACKBONE_KWARGS,
    stage1_objective_type=STAGE1_OBJECTIVE_TYPE,
    masking_type=STAGE1_MASKING_TYPE,
    mask_prob=STAGE1_MASK_PROB,
    mask_span_bins=STAGE1_MASK_SPAN_BINS,
    mask_replace_mode=STAGE1_MASK_REPLACE_MODE,
    reconstruction_head_type='linear',
    reconstruction_mlp_hidden_size=None,
)

print(STAGE1_CONFIG)


In [ ]:
# Stage-1 run/recover/resume.

if STAGE1_STATE_MODE == 'train':
    STAGE1_RUN_STATE = run_possm_training(
        cache_context=CACHE_CONTEXT,
        config=STAGE1_CONFIG,
        output_root=OUTPUT_ROOT,
        device=DEVICE,
    )
elif STAGE1_STATE_MODE == 'recover':
    stage1_checkpoint = resolve_possm_checkpoint_path(
        output_root=OUTPUT_ROOT,
        run_dir=STAGE1_RECOVERY_RUN_DIR,
        explicit_checkpoint_path=STAGE1_RECOVERY_EXPLICIT_CHECKPOINT_PATH,
    )
    STAGE1_RUN_STATE = recover_possm_run_state_from_checkpoint(
        cache_context=CACHE_CONTEXT,
        checkpoint_path=stage1_checkpoint,
        device=DEVICE,
    )
elif STAGE1_STATE_MODE == 'resume':
    stage1_checkpoint = resolve_possm_checkpoint_path(
        output_root=OUTPUT_ROOT,
        run_dir=STAGE1_RECOVERY_RUN_DIR,
        explicit_checkpoint_path=STAGE1_RECOVERY_EXPLICIT_CHECKPOINT_PATH,
    )
    STAGE1_RUN_STATE = recover_possm_run_state_from_checkpoint(
        cache_context=CACHE_CONTEXT,
        checkpoint_path=stage1_checkpoint,
        device=DEVICE,
    )
    STAGE1_RUN_STATE = resume_possm_training(
        run_state=STAGE1_RUN_STATE,
        additional_steps=STAGE1_RESUME_ADDITIONAL_STEPS,
        cache_context=CACHE_CONTEXT,
        device=DEVICE,
    )
else:
    raise ValueError('STAGE1_STATE_MODE must be one of {train, recover, resume}')

print('stage1 run_dir:', STAGE1_RUN_STATE['run_dir'])
print('stage1 final checkpoint:', STAGE1_RUN_STATE['checkpoint_path'])
print('stage1 best checkpoint:', STAGE1_RUN_STATE['best_checkpoint_path'])


In [ ]:
# Stage-1 metrics summary + loss curve.

stage1_progress_path = Path(STAGE1_RUN_STATE['progress_path'])
stage1_records = [json.loads(line) for line in stage1_progress_path.read_text().splitlines() if line.strip()]
train_rows = [r for r in stage1_records if r.get('event') == 'train']
val_rows = [r for r in stage1_records if r.get('event') == 'val']

print('train rows:', len(train_rows), '| val rows:', len(val_rows))
if train_rows:
    print('last train:', train_rows[-1])
if val_rows:
    print('last val:', val_rows[-1])

if train_rows:
    df_train = pd.DataFrame(train_rows)
    plt.figure(figsize=(8, 4))
    plt.plot(df_train['step'], df_train['loss'], label='train_mse')
    if val_rows:
        df_val = pd.DataFrame(val_rows)
        plt.plot(df_val['step'], df_val['loss'], label='val_mse')
    plt.title('Stage-1 POSSM Reconstruction Loss')
    plt.xlabel('step')
    plt.ylabel('MSE')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


In [ ]:
# Write Stage-1 run manifest sidecar.

stage1_manifest = {
    'stage': 'stage1_reconstruction',
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'seed': SEED,
    'data_mode': STAGE1_DATA_MODE,
    'feature_mode': FEATURE_MODE,
    'cache_root': str(CACHE_CONTEXT.cache_root),
    'cache_source_signature': getattr(CACHE_CONTEXT, 'source_cache_signature', None),
    'smoothing_provenance': smoothing_provenance,
    'checkpoint_best_path': str(STAGE1_RUN_STATE['best_checkpoint_path']),
    'checkpoint_final_path': str(STAGE1_RUN_STATE['checkpoint_path']),
    'run_dir': str(STAGE1_RUN_STATE['run_dir']),
}
stage1_manifest_path = Path(STAGE1_RUN_STATE['run_dir']) / 'run_manifest_stage1.json'
stage1_manifest_path.write_text(json.dumps(stage1_manifest, indent=2))
print('wrote:', stage1_manifest_path)


In [ ]:
# Stage-2 config + checkpoint handoff.

if STAGE2_STAGE1_CHECKPOINT_OVERRIDE is not None:
    STAGE2_STAGE1_CHECKPOINT = Path(STAGE2_STAGE1_CHECKPOINT_OVERRIDE)
else:
    STAGE2_STAGE1_CHECKPOINT = Path(STAGE1_RUN_STATE['best_checkpoint_path'])

STAGE2_CONFIG = POSSMFinetuneConfig(
    seed=SEED,
    mode='probe_frozen' if STAGE2_STATE_MODE == 'probe_frozen' else 'finetune_full',
    dataset='brain2text24',
    feature_mode=FEATURE_MODE,
    data_mode=STAGE1_DATA_MODE,
    boundary_key_mode=BOUNDARY_KEY_MODE,
    session_limit=8,
    target_session_count=4,
    batch_size=8,
    num_steps=400,
    budget_seconds=240,
    learning_rate=1e-3,
    encoder_learning_rate=3e-4,
    weight_decay=1e-2,
    max_grad_norm=1.0,
    checkpoint_every_steps=100,
    progress_every_steps=25,
    progress_every_seconds=15.0,
    gru_hidden_size=768,
    gru_num_layers=5,
    gru_dropout=0.4,
    conv_hidden_size=None,
    conv_kernel_size=14,
    conv_stride=4,
    conv_dropout=0.1,
)

print('STAGE2_STATE_MODE:', STAGE2_STATE_MODE)
print('STAGE2_STAGE1_CHECKPOINT:', STAGE2_STAGE1_CHECKPOINT)
print(STAGE2_CONFIG)


In [ ]:
# Stage-2 run (optional).

STAGE2_SUMMARY = None
if STAGE2_STATE_MODE == 'skip':
    print('Skipping stage-2 run.')
elif STAGE2_STATE_MODE in {'probe_frozen', 'finetune_full'}:
    STAGE2_SUMMARY = run_possm_phoneme_finetuning(
        checkpoint_path=STAGE2_STAGE1_CHECKPOINT,
        cache_root=Path(CACHE_CONTEXT.cache_root),
        output_root=OUTPUT_ROOT / 'phoneme_finetune',
        config=STAGE2_CONFIG,
        device=DEVICE,
    )
    print('stage2 run_dir:', STAGE2_SUMMARY['run_dir'])
    print('stage2 best checkpoint:', STAGE2_SUMMARY['checkpoint_best_path'])
else:
    raise ValueError('STAGE2_STATE_MODE must be one of {skip, probe_frozen, finetune_full}')


In [ ]:
# Stage-2 metrics summary + manifest.

if STAGE2_SUMMARY is None:
    print('No stage-2 summary (stage-2 skipped).')
else:
    metrics = dict(STAGE2_SUMMARY.get('metrics', {}))
    summary_row = {
        'mode': STAGE2_SUMMARY.get('mode'),
        'data_mode': STAGE2_SUMMARY.get('data_mode'),
        'feature_mode': STAGE2_SUMMARY.get('feature_mode'),
        'steps': STAGE2_SUMMARY.get('steps'),
        'val_ctc_bpphone': metrics.get('val_ctc_bpphone'),
        'val_phoneme_error_rate': metrics.get('val_phoneme_error_rate'),
        'best_val_ctc_bpphone': metrics.get('best_val_ctc_bpphone'),
        'best_val_phoneme_error_rate': metrics.get('best_val_phoneme_error_rate'),
        'checkpoint_best_path': STAGE2_SUMMARY.get('checkpoint_best_path'),
        'checkpoint_final_path': STAGE2_SUMMARY.get('checkpoint_final_path'),
    }
    display(pd.DataFrame([summary_row]))

    stage2_manifest = {
        'stage': 'stage2_phoneme_finetune',
        'created_utc': datetime.now(timezone.utc).isoformat(),
        'seed': SEED,
        'mode': STAGE2_SUMMARY.get('mode'),
        'data_mode': STAGE2_SUMMARY.get('data_mode'),
        'feature_mode': STAGE2_SUMMARY.get('feature_mode'),
        'cache_root': str(CACHE_CONTEXT.cache_root),
        'smoothing_provenance': smoothing_provenance,
        'stage1_checkpoint_path': str(STAGE2_STAGE1_CHECKPOINT),
        'checkpoint_best_path': STAGE2_SUMMARY.get('checkpoint_best_path'),
        'checkpoint_final_path': STAGE2_SUMMARY.get('checkpoint_final_path'),
        'run_dir': STAGE2_SUMMARY.get('run_dir'),
        'metrics': metrics,
    }
    stage2_manifest_path = Path(STAGE2_SUMMARY['run_dir']) / 'run_manifest_stage2.json'
    stage2_manifest_path.write_text(json.dumps(stage2_manifest, indent=2))
    print('wrote:', stage2_manifest_path)
